In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
SAVE_PATH = "/content/drive/MyDrive/final_project/"

required_files = [
    "voice_sentiment_model.h5",
    "voice_labels.npy",
    "behavioral_xgboost.pkl",
    "behavioral_lightgbm.pkl",
    "behavioral_catboost.cbm",
    "kmeans_model.pkl",
    "segment_scaler.pkl",
    "rl_agent.h5",
    "customer_segments.csv",
    "customer_recommendations.csv"
]

print("Checking all Phase 6 dependencies:\n")
for f in required_files:
    path   = SAVE_PATH + f
    status = "✅" if os.path.exists(path) else "❌"
    print(f"{status} {f}")

Checking all Phase 6 dependencies:

✅ voice_sentiment_model.h5
✅ voice_labels.npy
✅ behavioral_xgboost.pkl
✅ behavioral_lightgbm.pkl
✅ behavioral_catboost.cbm
✅ kmeans_model.pkl
✅ segment_scaler.pkl
✅ rl_agent.h5
✅ customer_segments.csv
✅ customer_recommendations.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, time
time.sleep(3)  # give drive time to sync

SAVE_PATH = "/content/drive/MyDrive/final_project/"
path = SAVE_PATH + "rl_agent.h5"

if os.path.exists(path):
    print("✅ rl_agent.h5 found!")
    print(f"Size: {os.path.getsize(path)/1024:.1f} KB")
else:
    print("❌ Still not found")
    print("\nListing all files:")
    for f in sorted(os.listdir(SAVE_PATH)):
        print(f"  {f}")

Mounted at /content/drive
✅ rl_agent.h5 found!
Size: 344.5 KB


In [ ]:
!pip install streamlit pyngrok plotly librosa catboost -q
print("Libraries installed ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 73.0 MB/s eta 0:00:00
Libraries installed ✅


In [ ]:
%%writefile app.py

import streamlit as st
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import joblib
import librosa
import tensorflow as tf
from tensorflow.keras.models import load_model
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────
st.set_page_config(
    page_title="Customer Engagement AI",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .main-title {
        font-size: 2.3rem;
        font-weight: bold;
        color: #2ecc71;
        text-align: center;
        padding: 1rem;
    }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────
# LOAD MODELS
# ─────────────────────────────────────────
BASE = "/content/drive/MyDrive/final_project/"

@st.cache_resource
def load_all_models():
    sentiment_model = load_model(
        BASE + "voice_sentiment_model.h5",
        compile=False
    )
    voice_labels = np.load(
        BASE + "voice_labels.npy",
        allow_pickle=True
    )

    xgb  = joblib.load(BASE + "behavioral_xgboost.pkl")
    lgbm = joblib.load(BASE + "behavioral_lightgbm.pkl")

    cat = CatBoostClassifier()
    cat.load_model(BASE + "behavioral_catboost.cbm")

    kmeans = joblib.load(BASE + "kmeans_model.pkl")
    scaler = joblib.load(BASE + "segment_scaler.pkl")

    rl_agent = load_model(
        BASE + "rl_agent.h5",
        compile=False
    )

    df   = pd.read_csv(BASE + "customer_segments.csv")
    recs = pd.read_csv(
        BASE + "customer_recommendations.csv"
    )

    feature_names = np.load(
        BASE + "feature_names.npy",
        allow_pickle=True
    )

    return (sentiment_model, voice_labels, xgb, lgbm,
            cat, kmeans, scaler, rl_agent, df, recs,
            feature_names)

(sentiment_model, voice_labels, xgb, lgbm, cat,
 kmeans, scaler, rl_agent, df, recs,
 feature_names) = load_all_models()

# ─────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────
def extract_audio_features(audio, sr):
    target = sr * 3
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    else:
        audio = audio[:target]

    mel    = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=128, fmax=8000
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mfcc   = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=40
    )
    chroma = librosa.feature.chroma_stft(
        y=audio, sr=sr, n_chroma=12
    )

    def norm(x):
        return (x - x.mean()) / (x.std() + 1e-6)

    return np.vstack([
        norm(mel_db), norm(mfcc), norm(chroma)
    ])

def predict_voice_sentiment(audio_file):
    audio, sr = librosa.load(
        audio_file, duration=3.0, offset=0.5
    )
    features = extract_audio_features(audio, sr)
    features = features[np.newaxis, ..., np.newaxis]
    prob = sentiment_model.predict(
        features, verbose=0
    )[0][0]
    label = voice_labels[1] if prob >= 0.5 \
        else voice_labels[0]
    return str(label), float(prob)

def predict_behavioral(input_df):
    xgb_prob  = xgb.predict_proba(input_df)[:,1][0]
    lgbm_prob = lgbm.predict_proba(input_df)[:,1][0]
    cat_prob  = cat.predict_proba(input_df)[:,1][0]
    return (xgb_prob + lgbm_prob + cat_prob) / 3

def get_segment(age, campaign, previous, pdays,
                emp_var_rate, cons_conf_idx,
                cons_price_idx, euribor3m,
                nr_employed, sentiment_score,
                behavioral_score):
    features = np.array([[
        age, campaign, previous, pdays,
        emp_var_rate, cons_conf_idx,
        cons_price_idx, euribor3m, nr_employed,
        sentiment_score, behavioral_score
    ]])
    features_scaled = scaler.transform(features)
    cluster = kmeans.predict(features_scaled)[0]

    cluster_map = {
        0: '🔴 Cold',
        1: '🟢 Hot Lead',
        2: '🔕 Do Not Disturb',
        3: '🔵 Neutral',
        4: '🟡 Warm Lead'
    }
    return cluster_map.get(cluster, '🔵 Neutral')

def get_rl_action(sentiment_score, behavioral_score,
                  age, campaign, previous,
                  emp_var_rate, cons_conf_idx):
    state = np.array([[
        sentiment_score,
        behavioral_score,
        age      / 100.0,
        campaign / 50.0,
        previous / 10.0,
        emp_var_rate / 5.0,
        cons_conf_idx / 100.0
    ]], dtype=np.float32)

    q_values = rl_agent.predict(state, verbose=0)[0]
    action   = np.argmax(q_values)

    actions = {
        0: '📞 Call Now',
        1: '📱 Send SMS',
        2: '📧 Send Email',
        3: '⏳ Wait & Nurture',
        4: '🔕 Do Not Disturb'
    }
    return actions[action], q_values

# ─────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────
st.sidebar.markdown("## 🤖 Navigation")
page = st.sidebar.radio(
    "Go to",
    ["🏠 Home",
     "🎤 Voice Sentiment",
     "👤 Customer Predictor",
     "📊 Analytics"]
)

st.sidebar.markdown("---")
st.sidebar.markdown("### 📊 Quick Stats")
st.sidebar.metric("Total Customers", "41,188")
st.sidebar.metric("Voice Accuracy",  "69.74%")
st.sidebar.metric("Behavioral AUC",  "81.29%")
st.sidebar.metric("RL Accuracy",     "95.86%")

# ─────────────────────────────────────────
# PAGE 1 — HOME
# ─────────────────────────────────────────
if page == "🏠 Home":
    st.markdown(
        '<div class="main-title">🤖 Multimodal Sentiment-Driven'
        ' Customer Engagement Optimization</div>',
        unsafe_allow_html=True
    )
    st.markdown(
        "<p style='text-align:center; color:#95a5a6;'>"
        "Powered by CNN-LSTM-Attention · XGBoost · "
        "K-Means · Deep Q-Network</p>",
        unsafe_allow_html=True
    )
    st.markdown("---")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("🎤 Voice Accuracy", "69.74%",
              "CNN-LSTM-Attention")
    c2.metric("📊 Behavioral AUC", "81.29%",
              "Ensemble Model")
    c3.metric("🎯 Segments",       "5", "K-Means")
    c4.metric("🤖 RL Accuracy",    "95.86%",
              "DQN Agent")

    st.markdown("---")
    st.markdown("## 🔄 Project Pipeline")
    cols = st.columns(5)
    pipeline = [
        ("🎤", "Phase 2", "Voice Sentiment",
         "CNN-LSTM-Attention", "#2ecc71"),
        ("📊", "Phase 3", "Behavioral Model",
         "XGB+LGBM+CatBoost",  "#3498db"),
        ("🎯", "Phase 4", "Segmentation",
         "K-Means K=5",         "#f1c40f"),
        ("🤖", "Phase 5", "RL Agent",
         "Deep Q-Network",      "#e74c3c"),
        ("🖥️", "Phase 6", "Dashboard",
         "Streamlit",           "#9b59b6"),
    ]
    for col, (icon, phase, title, model, color) \
            in zip(cols, pipeline):
        col.markdown(
            f"<div style='text-align:center;"
            f"border:2px solid {color};"
            f"border-radius:10px;padding:1rem;'>"
            f"<div style='font-size:2rem'>{icon}</div>"
            f"<div style='color:{color};"
            f"font-weight:bold'>{phase}</div>"
            f"<div style='font-size:0.9rem'>{title}</div>"
            f"<div style='color:#95a5a6;"
            f"font-size:0.8rem'>{model}</div>"
            f"</div>",
            unsafe_allow_html=True
        )

    st.markdown("---")
    st.markdown("## 👥 Customer Segments Overview")
    seg_counts = df['segment'].value_counts()\
        .reset_index()
    seg_counts.columns = ['Segment', 'Count']

    colors_map = {
        '🟢 Hot Lead'       : '#2ecc71',
        '🟡 Warm Lead'      : '#f1c40f',
        '🔵 Neutral'        : '#3498db',
        '🔴 Cold'           : '#e74c3c',
        '🔕 Do Not Disturb' : '#95a5a6'
    }

    c1, c2 = st.columns(2)
    with c1:
        fig = px.pie(
            seg_counts, values='Count',
            names='Segment', color='Segment',
            color_discrete_map=colors_map,
            title='Segment Distribution'
        )
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        fig = px.bar(
            seg_counts, x='Segment', y='Count',
            color='Segment',
            color_discrete_map=colors_map,
            title='Customer Count Per Segment'
        )
        st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────
# PAGE 2 — VOICE SENTIMENT
# ─────────────────────────────────────────
elif page == "🎤 Voice Sentiment":
    st.title("🎤 Voice Sentiment Analysis")
    st.markdown(
        "Upload a customer call audio file to detect "
        "engagement sentiment in real time."
    )
    st.markdown("---")

    uploaded = st.file_uploader(
        "Upload WAV Audio File", type=['wav']
    )

    if uploaded is not None:
        st.audio(uploaded)
        with st.spinner("Analyzing voice sentiment..."):
            import tempfile, os
            with tempfile.NamedTemporaryFile(
                delete=False, suffix='.wav'
            ) as tmp:
                tmp.write(uploaded.read())
                tmp_path = tmp.name
            label, prob = predict_voice_sentiment(
                tmp_path
            )
            os.unlink(tmp_path)

        st.markdown("---")
        st.markdown("## 🎯 Prediction Result")
        c1, c2 = st.columns(2)

        with c1:
            if label == "Positive":
                st.success(f"## {label} 😊")
                st.markdown(
                    "Customer sounds **positive and "
                    "receptive**. Good time to present offer."
                )
            else:
                st.error(f"## {label} 😟")
                st.markdown(
                    "Customer sounds **negative or "
                    "resistant**. Handle with care."
                )
            st.metric("Confidence Score",
                      f"{prob*100:.1f}%")

        with c2:
            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=prob * 100,
                title={'text': "Positive Sentiment Score"},
                gauge={
                    'axis': {'range': [0, 100]},
                    'bar': {'color': "#2ecc71"},
                    'steps': [
                        {'range': [0, 50],
                         'color': "#e74c3c"},
                        {'range': [50, 100],
                         'color': "#2ecc71"}
                    ],
                }
            ))
            fig.update_layout(height=300)
            st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("👆 Upload a WAV audio file to get started.")
        st.markdown("---")
        st.markdown("## 🧠 Model Information")
        c1, c2, c3 = st.columns(3)
        c1.metric("Architecture", "CNN-LSTM-Attention")
        c2.metric("Dataset", "CREMA-D")
        c3.metric("Accuracy", "69.74%")

# ─────────────────────────────────────────
# PAGE 3 — CUSTOMER PREDICTOR
# ─────────────────────────────────────────
elif page == "👤 Customer Predictor":
    st.title("👤 Customer Engagement Predictor")
    st.markdown(
        "Enter customer details to get complete "
        "engagement analysis and recommended action."
    )
    st.markdown("---")

    st.markdown("### 🎤 Optional — Upload Customer Audio")
    audio_file = st.file_uploader(
        "Upload call recording (WAV)",
        type=['wav'], key="predictor_audio"
    )

    sentiment_score = 0.5
    sentiment_label = "No audio uploaded (default: Neutral)"

    if audio_file:
        with st.spinner("Analyzing audio..."):
            import tempfile, os
            with tempfile.NamedTemporaryFile(
                delete=False, suffix='.wav'
            ) as tmp:
                tmp.write(audio_file.read())
                tmp_path = tmp.name
            label, prob = predict_voice_sentiment(tmp_path)
            os.unlink(tmp_path)
            sentiment_score = prob
            sentiment_label = f"{label} ({prob*100:.1f}%)"
        st.success(f"Detected sentiment: {sentiment_label}")

    st.markdown("---")

    with st.form("customer_form"):
        st.markdown("### 📋 Customer & Call Details")
        c1, c2, c3 = st.columns(3)

        with c1:
            age      = st.slider("Age", 18, 95, 38)
            campaign = st.slider(
                "Contacts This Campaign", 1, 30, 2
            )
            previous = st.slider(
                "Previous Contacts", 0, 10, 0
            )
            pdays    = st.slider(
                "Days Since Last Contact "
                "(999=never)", 0, 999, 999
            )

        with c2:
            poutcome = st.selectbox(
                "Previous Outcome",
                ["nonexistent", "failure", "success"]
            )
            contact = st.selectbox(
                "Contact Type",
                ["cellular", "telephone"]
            )
            month = st.selectbox(
                "Month",
                ["jan","feb","mar","apr","may","jun",
                 "jul","aug","sep","oct","nov","dec"]
            )
            day_of_week = st.selectbox(
                "Day of Week",
                ["mon","tue","wed","thu","fri"]
            )

        with c3:
            job = st.selectbox(
                "Job",
                ["admin.","blue-collar","entrepreneur",
                 "housemaid","management","retired",
                 "self-employed","services","student",
                 "technician","unemployed","unknown"]
            )
            marital = st.selectbox(
                "Marital Status",
                ["married","single","divorced"]
            )
            education = st.selectbox(
                "Education",
                ["basic.4y","basic.6y","basic.9y",
                 "high.school","illiterate",
                 "professional.course",
                 "university.degree","unknown"]
            )
            housing = st.selectbox(
                "Housing Loan", ["no","yes","unknown"]
            )

        st.markdown("### 📈 Economic Indicators")
        c4, c5, c6 = st.columns(3)
        with c4:
            emp_var_rate = st.slider(
                "Employment Variation Rate",
                -3.5, 1.5, 1.1, step=0.1
            )
        with c5:
            cons_conf_idx = st.slider(
                "Consumer Confidence Index",
                -51.0, -26.0, -36.0, step=0.5
            )
        with c6:
            euribor3m = st.slider(
                "Euribor 3 Month Rate",
                0.5, 5.5, 4.8, step=0.1
            )

        submitted = st.form_submit_button(
            "🔍 Predict Engagement",
            use_container_width=True
        )

    if submitted:
        # Build feature row matching Phase 3 OHE
        row = {c: 0 for c in feature_names}
        row['age']           = age
        row['campaign']      = campaign
        row['pdays']         = pdays
        row['previous']      = previous
        row['emp.var.rate']  = emp_var_rate
        row['cons.price.idx']= 93.5
        row['cons.conf.idx'] = cons_conf_idx
        row['euribor3m']     = euribor3m
        row['nr.employed']   = 5191.0

        for col in feature_names:
            if col == f'job_{job}':
                row[col] = 1
            if col == f'marital_{marital}':
                row[col] = 1
            if col == f'education_{education}':
                row[col] = 1
            if col == f'housing_{housing}':
                row[col] = 1
            if col == f'contact_{contact}':
                row[col] = 1
            if col == f'month_{month}':
                row[col] = 1
            if col == f'day_of_week_{day_of_week}':
                row[col] = 1
            if col == f'poutcome_{poutcome}':
                row[col] = 1

        input_df = pd.DataFrame([row])[
            list(feature_names)
        ]

        with st.spinner("Analyzing customer..."):
            behavioral_score = predict_behavioral(
                input_df
            )
            segment = get_segment(
                age, campaign, previous, pdays,
                emp_var_rate, cons_conf_idx,
                93.5, euribor3m, 5191.0,
                sentiment_score, behavioral_score
            )
            action, q_values = get_rl_action(
                sentiment_score, behavioral_score,
                age, campaign, previous,
                emp_var_rate, cons_conf_idx
            )

        st.markdown("---")
        st.markdown("## 🎯 Engagement Analysis Results")

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Engagement Likelihood",
                  f"{behavioral_score*100:.1f}%",
                  "Behavioral Score")
        c2.metric("Voice Sentiment", sentiment_label)
        c3.metric("Customer Segment", segment)
        c4.metric("Recommended Action", action)

        st.markdown("---")
        action_colors = {
            '📞 Call Now'       : '#2ecc71',
            '📱 Send SMS'       : '#f1c40f',
            '📧 Send Email'     : '#3498db',
            '⏳ Wait & Nurture' : '#e74c3c',
            '🔕 Do Not Disturb' : '#95a5a6'
        }
        color = action_colors.get(action, '#2ecc71')
        st.markdown(
            f"<div style='background:{color};"
            f"color:white;padding:2rem;"
            f"border-radius:15px;text-align:center;"
            f"font-size:2rem;font-weight:bold;'>"
            f"Recommended Action: {action}</div>",
            unsafe_allow_html=True
        )

        st.markdown("---")
        st.markdown("### 🧠 RL Agent Q-Values")
        actions_list = ['📞 Call','📱 SMS','📧 Email',
                        '⏳ Wait','🔕 DND']
        fig = px.bar(
            x=actions_list, y=q_values,
            color=actions_list,
            title="Q-Values (higher = better)",
            labels={'x':'Action','y':'Q-Value'}
        )
        st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────
# PAGE 4 — ANALYTICS
# ─────────────────────────────────────────
elif page == "📊 Analytics":
    st.title("📊 Analytics Dashboard")
    st.markdown("---")

    colors_map = {
        '🟢 Hot Lead'       : '#2ecc71',
        '🟡 Warm Lead'      : '#f1c40f',
        '🔵 Neutral'        : '#3498db',
        '🔴 Cold'           : '#e74c3c',
        '🔕 Do Not Disturb' : '#95a5a6'
    }

    st.markdown("## 👥 Customer Segments")
    seg_counts = df['segment'].value_counts()\
        .reset_index()
    seg_counts.columns = ['Segment', 'Count']

    c1, c2 = st.columns(2)
    with c1:
        fig = px.pie(
            seg_counts, values='Count',
            names='Segment', color='Segment',
            color_discrete_map=colors_map,
            title='Segment Distribution'
        )
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        fig = px.bar(
            seg_counts, x='Segment', y='Count',
            color='Segment',
            color_discrete_map=colors_map,
            title='Customer Count Per Segment'
        )
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    st.markdown("## 🤖 RL Agent Recommendations")
    if 'recommended_action' in recs.columns:
        action_counts = recs['recommended_action']\
            .value_counts().reset_index()
        action_counts.columns = ['Action', 'Count']
        fig = px.bar(
            action_counts, x='Action', y='Count',
            color='Action',
            title='Recommended Actions Distribution'
        )
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    st.markdown("## 📈 Model Performance")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Voice Accuracy",      "69.74%")
    c2.metric("Voice AUC",           "0.7543")
    c3.metric("Behavioral AUC",      "81.29%")
    c4.metric("RL Agent Accuracy",   "95.86%")

    st.markdown("---")
    st.markdown("## 🔍 Sentiment vs Behavioral Score")
    fig = px.scatter(
        df, x='behavioral_score', y='sentiment_score',
        color='segment', color_discrete_map=colors_map,
        title='Customer Distribution', opacity=0.5
    )
    st.plotly_chart(fig, use_container_width=True)

print("app.py written ✅")

Writing app.py


In [ ]:
from pyngrok import ngrok
import subprocess
import time

ngrok.kill()
ngrok.set_auth_token("3F4YumkwiYxAmc5aD2gQ6OfsE2P_EiAfiyC4bcmJ3KMynZKq")

process = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

public_url = ngrok.connect(8501)
print(f"\n🚀 Dashboard is live!")
print(f"👉 Open this URL: {public_url}")


🚀 Dashboard is live!
👉 Open this URL: NgrokTunnel: "https://cycling-fracture-islamic.ngrok-free.dev" -> "http://localhost:8501"
